In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0. IMPORTS — install if missing:
!pip install lime shap catboost dice-ml scikit-learn
# ─────────────────────────────────────────────
import lime
import lime.lime_tabular
import shap

# ── Minimal reproducible placeholder (DELETE after replacing above) ──
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.1 MB/s eta 0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=d62fe91f2900ef49b4f3981e570cf8a5de20e3da6e85b9f1b68e1ca851173ade
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [ ]:
from imblearn.over_sampling import SMOTENC
from collections import Counter
from sklearn.preprocessing import StandardScaler

def Apply_SMOTE_NC(X, y):
  # Identify the indices of categorical features
  categorical_features = [col for col in X.columns if X[col].nunique() < 5]

  # Apply SMOTENC to the training data
  smote_nc = SMOTENC(categorical_features=categorical_features, random_state=42)
  X_res_smote_nc, y_res_smote_nc = smote_nc.fit_resample(X, y)

  # Check the class distribution after SMOTENC
  print(f"Resampled class distribution (SMOTENC): {Counter(y_res_smote_nc)}")
  #df = pd.concat([X_res_smote_nc, y_res_smote_nc], axis=1)
  return X_res_smote_nc, y_res_smote_nc

from sklearn.preprocessing import StandardScaler
def Normalize_data(df):
  columns_to_normalize = [col for col in df.columns if df[col].nunique() > 10]
  # Standardization (Z-score normalization)
  scaler = StandardScaler()
  df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
  return df

In [ ]:
np.random.seed(42)
import pandas as pd
from sklearn.preprocessing import StandardScaler # Import StandardScaler

# --- USER INPUT REQUIRED ---
# Please specify the path to your CSV file:
csv_file_path = "/content/df_encoded_dropped_outlhandled_new.csv"
# Please specify the name of the target column in your CSV:
target_column_name = "Attrition"
# ---------------------------

# Load the data
df = pd.read_csv(csv_file_path)

# Separate features (X) and target (y)
X_all = df.drop(columns=[target_column_name]).values
y_all = df[target_column_name].values
feature_names = df.drop(columns=[target_column_name]).columns.tolist()

# Determine class names (assuming binary classification)
unique_classes = np.unique(y_all)
class_names = [f"Class {c}" for c in unique_classes]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42
)

# Convert X_train and X_test to DataFrames for SMOTE-NC and Normalization
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

# Apply SMOTE-NC to training data
print("Applying SMOTE-NC to training data...")
X_train_res_df, y_train_res = Apply_SMOTE_NC(X_train_df, y_train)

# Identify columns to normalize (numerical columns with more than 10 unique values)
columns_to_normalize = [col for col in X_train_res_df.columns if X_train_res_df[col].nunique() > 10]

# Initialize and fit StandardScaler on the resampled training data
scaler = StandardScaler()
X_train_res_df[columns_to_normalize] = scaler.fit_transform(X_train_res_df[columns_to_normalize])

# Transform the test data using the *same* scaler fitted on the training data
X_test_df[columns_to_normalize] = scaler.transform(X_test_df[columns_to_normalize])

# Convert processed DataFrames back to NumPy arrays for model training
X_train_processed = X_train_res_df.values
X_test_processed = X_test_df.values

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_processed, y_train_res)
# ── End placeholder ──────────────────────────

print(f"Model test accuracy: {accuracy_score(y_test, model.predict(X_test_processed)):.4f}")

Applying SMOTE-NC to training data...
Resampled class distribution (SMOTENC): Counter({np.float64(0.0): 1181, np.float64(1.0): 1181})
Model test accuracy: 0.9048


In [ ]:
# Convert to numpy arrays for consistency
X_train_np = np.array(X_train_processed)
X_test_np  = np.array(X_test_processed)

# Select evaluation instances: true-positive attrition cases
# (instances where model correctly predicts attrition = class 1)
y_pred = model.predict(X_test_np)
tp_mask = (y_pred == 1) & (y_test == 1) # Assuming class 1 is the positive class
eval_instances = X_test_np[tp_mask][:30]   # use up to 30 instances
print(f"Using {len(eval_instances)} true-positive instances for evaluation")

Using 29 true-positive instances for evaluation


In [ ]:
# ─────────────────────────────────────────────
# 2. SETUP EXPLAINERS
# ─────────────────────────────────────────────

TOP_N = 12   # same threshold as your paper

# LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=feature_names,
    class_names=class_names, # Use inferred class_names
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

# SHAP explainer
shap_explainer = shap.LinearExplainer(model, X_train_np)


# ─────────────────────────────────────────────
# HELPER: get top-N features from LIME for one instance
# ─────────────────────────────────────────────
def get_lime_top_features(instance, n=TOP_N, num_samples=500, seed=None):
    """Returns a frozenset of the top-n LIME feature names."""
    if seed is not None:
        np.random.seed(seed)
    exp = lime_explainer.explain_instance(
        instance,
        model.predict_proba,
        num_features=n,
        num_samples=num_samples,
        labels=[1]
    )
    feats = [feat for feat, _ in exp.as_list(label=1)]
    # LIME returns feature conditions like "feature_3 > 0.5"
    # extract the base feature name
    base_feats = set()
    for f in feats:
        for fname in feature_names:
            if fname in f:
                base_feats.add(fname)
                break
    return frozenset(base_feats)

In [ ]:
# ─────────────────────────────────────────────
# HELPER: get top-N features from SHAP for one instance
# ─────────────────────────────────────────────
def get_shap_top_features(instance, n=TOP_N):
    """Returns a frozenset of the top-n SHAP feature names (class 1)."""
    sv = shap_explainer.shap_values(instance.reshape(1, -1))
    # For LinearExplainer, sv is typically a single array or a list of two for binary classification
    if isinstance(sv, list):
        vals = np.abs(sv[1]).flatten() # Flatten to ensure 1D array
    else:
        vals = np.abs(sv).flatten() # Flatten to ensure 1D array
    top_idx = np.argsort(vals)[-n:]
    return frozenset(feature_names[i] for i in top_idx)

In [ ]:
# ─────────────────────────────────────────────
# HELPER: LSC-XAI intersection features
# ─────────────────────────────────────────────

# Define feasible features (same as your paper)
# NOTE: These are placeholder features and likely need to be updated to match your CSV's feature names.
FEASIBLE_FEATURES = {
    "Shift", "OverTime", "EnvironmentSatisfaction",
    "JobInvolvement", "JobLevel", "TrainingTimesLastYear"
}
# Map to your actual feature names — update this set to match yours exactly

def get_lsc_features(instance, n=TOP_N):
    """
    Returns the LSC-XAI feature set:
    intersection of LIME and SHAP, filtered to feasible features.
    """
    lime_feats = get_lime_top_features(instance, n=n)
    shap_feats = get_shap_top_features(instance, n=n)
    intersection = lime_feats & shap_feats
    # filter to feasible (if any match; else return full intersection)
    feasible_intersection = intersection & FEASIBLE_FEATURES
    return feasible_intersection if feasible_intersection else intersection

In [ ]:
# ─────────────────────────────────────────────
# METRIC 1: EXPLANATION STABILITY
# ─────────────────────────────────────────────
# Run each explainer N_RUNS times on the same instance with different seeds.
# Compute average pairwise Jaccard similarity of the returned feature sets.
# Higher Jaccard = more stable.
#
# Jaccard(A, B) = |A ∩ B| / |A ∪ B|

def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    return len(set_a & set_b) / len(set_a | set_b)

def stability_score(feature_sets):
    """
    Mean pairwise Jaccard similarity across a list of feature sets.
    """
    n = len(feature_sets)
    if n < 2:
        return 1.0
    scores = []
    for i in range(n):
        for j in range(i + 1, n):
            scores.append(jaccard(feature_sets[i], feature_sets[j]))
    return np.mean(scores)

N_RUNS = 10
print("\n── Computing Metric 1: Stability ──")
lime_stab, shap_stab, lsc_stab = [], [], []

for idx, inst in enumerate(eval_instances):
    # LIME stability: run with different seeds
    lime_runs = [get_lime_top_features(inst, seed=s) for s in range(N_RUNS)]
    lime_stab.append(stability_score(lime_runs))

    # SHAP stability: deterministic, so stability = 1.0 by definition
    # (TreeSHAP is exact, not sampled)
    shap_stab.append(1.0)

    # LSC-XAI stability: intersection reduces LIME's variance
    lsc_runs = []
    for s in range(N_RUNS):
        np.random.seed(s)
        lime_f = get_lime_top_features(inst, seed=s)

        # Fix: Ensure vals is a 1-D array before argsort to correctly get top_idx
        sv = shap_explainer.shap_values(inst.reshape(1, -1))
        if isinstance(sv, list):
            vals = np.abs(sv[1]).flatten() # Flatten here
        else:
            vals = np.abs(sv).flatten() # Flatten here

        top_idx = np.argsort(vals)[-TOP_N:]
        shap_f = frozenset(feature_names[i] for i in top_idx)

        inter  = lime_f & shap_f
        feasible = inter & FEASIBLE_FEATURES
        lsc_runs.append(feasible if feasible else inter)
    lsc_stab.append(stability_score(lsc_runs))

    if (idx + 1) % 5 == 0:
        print(f"  Processed {idx+1}/{len(eval_instances)} instances...")

stability_results = {
    "LIME":            np.mean(lime_stab),
    "SHAP":            np.mean(shap_stab),
    "Counterfactuals": 0.78,   # ← replace with your CF variance measurement
    "LSC-XAI":         np.mean(lsc_stab),
}
print(f"  LIME stability:    {stability_results['LIME']:.4f}")
print(f"  SHAP stability:    {stability_results['SHAP']:.4f}")
print(f"  LSC-XAI stability: {stability_results['LSC-XAI']:.4f}")


── Computing Metric 1: Stability ──
  Processed 5/29 instances...
  Processed 10/29 instances...
  Processed 15/29 instances...
  Processed 20/29 instances...
  Processed 25/29 instances...
  LIME stability:    0.7373
  SHAP stability:    1.0000
  LSC-XAI stability: 0.9369


In [ ]:
# ─────────────────────────────────────────────
# METRIC 2: FIDELITY
# ─────────────────────────────────────────────
# Fidelity = how accurately a LOCAL surrogate model (trained only on the
# explanation's features) mimics the black-box model on nearby samples.
#
# Protocol:
#   1. Take the top-N features from each method.
#   2. Train a Logistic Regression on X_train using ONLY those features.
#   3. Measure its accuracy on X_test vs. the CatBoost predictions (not y_test).
#      (We're measuring how well the explanation reproduces the model, not the truth.)
#
# Higher fidelity = the explanation better captures what the model does.

print("\n── Computing Metric 2: Fidelity ──")

def fidelity_of_feature_set(feature_indices, X_tr, X_te, model_preds_train, model_preds_test):
    """
    Train a Logistic Regression on feature_indices subset.
    Return accuracy vs. black-box model predictions on test set.
    """
    if not feature_indices:
        return 0.0
    X_tr_sub = X_tr[:, list(feature_indices)]
    X_te_sub = X_te[:, list(feature_indices)]
    lr = LogisticRegression(max_iter=500, random_state=42)
    lr.fit(X_tr_sub, model_preds_train)
    surr_preds = lr.predict(X_te_sub)
    return accuracy_score(model_preds_test, surr_preds)

# Get black-box predictions
bb_train = model.predict(X_train_np)
bb_test  = model.predict(X_test_np)

# For fidelity we use the globally most common top features across all instances
# (this is the standard approach for global fidelity estimation)

all_lime_feats_fid, all_shap_feats_fid, all_lsc_feats_fid = set(), set(), set()

for inst in eval_instances:
    lf = get_lime_top_features(inst)
    sf = get_shap_top_features(inst)
    lsc_f = (lf & sf) or lf   # fallback to lime if intersection empty
    all_lime_feats_fid.update(lf)
    all_shap_feats_fid.update(sf)
    all_lsc_feats_fid.update(lsc_f)

# Get feature indices
def feats_to_idx(feat_set):
    return [feature_names.index(f) for f in feat_set if f in feature_names]

lime_idx = feats_to_idx(all_lime_feats_fid)[:TOP_N]
shap_idx = feats_to_idx(all_shap_feats_fid)[:TOP_N]
lsc_idx  = feats_to_idx(all_lsc_feats_fid)[:TOP_N]

fidelity_results = {
    "LIME":            fidelity_of_feature_set(lime_idx, X_train_np, X_test_np, bb_train, bb_test),
    "SHAP":            fidelity_of_feature_set(shap_idx, X_train_np, X_test_np, bb_train, bb_test),
    "Counterfactuals": None,   # counterfactuals don't produce a feature ranking natively
    "LSC-XAI":         fidelity_of_feature_set(lsc_idx,  X_train_np, X_test_np, bb_train, bb_test),
}
print(f"  LIME fidelity:    {fidelity_results['LIME']:.4f}")
print(f"  SHAP fidelity:    {fidelity_results['SHAP']:.4f}")
print(f"  LSC-XAI fidelity: {fidelity_results['LSC-XAI']:.4f}")


── Computing Metric 2: Fidelity ──
  LIME fidelity:    0.7679
  SHAP fidelity:    0.7946
  LSC-XAI fidelity: 0.7649


In [ ]:
# ─────────────────────────────────────────────
# METRIC 3: SPARSITY
# ─────────────────────────────────────────────
# Sparsity = average number of features in the final explanation.
# LOWER is better (more focused, more actionable for HR managers).
#
# Formula used: sparsity = 1 - (|features_used| / |total_features|)
# (higher value = sparser)

print("\n── Computing Metric 3: Sparsity ──")
total_feats = len(feature_names)

lime_sizes, shap_sizes, lsc_sizes = [], [], []
for inst in eval_instances:
    lime_sizes.append(len(get_lime_top_features(inst)))
    shap_sizes.append(len(get_shap_top_features(inst)))
    lsc_f = get_lsc_features(inst)
    lsc_sizes.append(len(lsc_f))

def sparsity_score(sizes, total):
    return 1 - (np.mean(sizes) / total)

sparsity_results = {
    "LIME":            sparsity_score(lime_sizes, total_feats),
    "SHAP":            sparsity_score(shap_sizes, total_feats),
    "Counterfactuals": 1 - (np.mean([4, 5, 3, 6, 4]) / total_feats),  # ← replace with CF feature count
    "LSC-XAI":         sparsity_score(lsc_sizes, total_feats),
}
mean_lsc_feats = np.mean(lsc_sizes)
print(f"  LIME avg features:    {np.mean(lime_sizes):.1f}")
print(f"  SHAP avg features:    {np.mean(shap_sizes):.1f}")
print(f"  LSC-XAI avg features: {mean_lsc_feats:.1f}")
print(f"  LSC-XAI sparsity:     {sparsity_results['LSC-XAI']:.4f}")


── Computing Metric 3: Sparsity ──
  LIME avg features:    11.7
  SHAP avg features:    12.0
  LSC-XAI avg features: 3.3
  LSC-XAI sparsity:     0.8966


In [ ]:
# ─────────────────────────────────────────────
# METRIC 4: COUNTERFACTUAL PROXIMITY & PLAUSIBILITY
# ─────────────────────────────────────────────
# Proximity  = L1 distance between original instance and counterfactual
#              (smaller = fewer/smaller changes needed = better)
# Plausibility = fraction of counterfactuals that land within the
#                training data distribution (within 2 std of feature mean)
#              (higher = more realistic suggestions)
#
# This is computed only for the counterfactual component.
# For LSC-XAI, the feasibility filter should IMPROVE plausibility.

print("\n── Computing Metric 4: Counterfactual Proximity & Plausibility ──")

# Compute training data statistics for plausibility check
train_mean = X_train_np.mean(axis=0)
train_std  = X_train_np.std(axis=0) + 1e-8

def generate_simple_cf(instance, model, feasible_idx, n_attempts=50):
    """
    Simple random-search counterfactual generator.
    Perturbs only feasible features until model flips prediction.
    Returns the closest counterfactual found, or None.
    """
    orig_pred = model.predict(instance.reshape(1, -1))[0]
    best_cf = None
    best_dist = np.inf

    for _ in range(n_attempts):
        candidate = instance.copy()
        for fi in feasible_idx:
            # perturb by up to 1 std in a random direction
            candidate[fi] += np.random.uniform(-1.5, 1.5) * train_std[fi]
        new_pred = model.predict(candidate.reshape(1, -1))[0]
        if new_pred != orig_pred:
            dist = np.sum(np.abs(candidate - instance))
            if dist < best_dist:
                best_dist = dist
                best_cf = candidate
    return best_cf, best_dist

def is_plausible(cf, mean, std, z_thresh=2.0):
    """Returns True if all features of cf are within z_thresh std of training mean."""
    return bool(np.all(np.abs(cf - mean) / std < z_thresh))

# Get feasible feature indices
feasible_idx = [i for i, f in enumerate(feature_names) if f in FEASIBLE_FEATURES]
# If no match (placeholder data), use first 6 features
if not feasible_idx:
    feasible_idx = list(range(6))

all_feat_idx = list(range(len(feature_names)))[:TOP_N]

prox_standalone, plaus_standalone = [], []
prox_lsc,        plaus_lsc        = [], []

for inst in eval_instances[:15]:   # limit to 15 for speed
    # Standalone counterfactual: any feature can change
    cf_sa, dist_sa = generate_simple_cf(inst, model, all_feat_idx)
    if cf_sa is not None:
        prox_standalone.append(dist_sa)
        plaus_standalone.append(float(is_plausible(cf_sa, train_mean, train_std)))

    # LSC-XAI counterfactual: only feasible features change
    cf_lsc, dist_lsc = generate_simple_cf(inst, model, feasible_idx)
    if cf_lsc is not None:
        prox_lsc.append(dist_lsc)
        plaus_lsc.append(float(is_plausible(cf_lsc, train_mean, train_std)))

cf_results = {
    "standalone": {
        "proximity":    np.mean(prox_standalone) if prox_standalone else float("nan"),
        "plausibility": np.mean(plaus_standalone) if plaus_standalone else float("nan"),
    },
    "lsc_xai": {
        "proximity":    np.mean(prox_lsc) if prox_lsc else float("nan"),
        "plausibility": np.mean(plaus_lsc) if plaus_lsc else float("nan"),
    }
}
print(f"  Standalone CF proximity:    {cf_results['standalone']['proximity']:.4f}")
print(f"  LSC-XAI CF proximity:       {cf_results['lsc_xai']['proximity']:.4f}")
print(f"  Standalone CF plausibility: {cf_results['standalone']['plausibility']:.4f}")
print(f"  LSC-XAI CF plausibility:    {cf_results['lsc_xai']['plausibility']:.4f}")


── Computing Metric 4: Counterfactual Proximity & Plausibility ──
  Standalone CF proximity:    7.0268
  LSC-XAI CF proximity:       2.9384
  Standalone CF plausibility: 0.0769
  LSC-XAI CF plausibility:    0.1429
